# 14 — Revisión posterior, sin reentrenamiento

Lee las tablas ya guardadas por 08b y los paneles factoriales de 12. No ejecuta esos notebooks, no carga checkpoints y no modifica sus resultados. Compara InterGATE y q con el mismo selector y presupuesto dirigido; separa filas dirigidas y pares únicos, y calcula BH por ámbito externo sobre los pares del núcleo estricto. Son análisis adicionales post hoc. La primera comparación es descriptiva; el bootstrap por pacientes es opcional.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'intergate').is_dir() and (p / 'configs/protocol_v8.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abra el notebook desde la raíz de InterGATE o desde notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from intergate.review_posthoc import run_review
from intergate.publication_style import load_style, apply_style, prepare_figure
RESULTS_ROOT = PROJECT_ROOT / 'results_v8'
OUTPUT_DIR = RESULTS_ROOT / '14_review_posthoc'
STYLE = apply_style(load_style(PROJECT_ROOT))
print('Entrada:', RESULTS_ROOT)
print('Salida nueva:', OUTPUT_DIR)


In [ ]:
tables = run_review(RESULTS_ROOT, OUTPUT_DIR)
for name in ('strict_core_reciprocity', 'strict_core_concordance_by_unit', 'q_comparator_input_status', 'intergate_vs_q_external_differences'):
    if name in tables:
        print('\n' + name)
        display(tables[name])
pair_effects = tables['strict_core_unique_pair_effects']
columns = ['pair_gene_a', 'pair_gene_b', 'n_directed_rows', 'n_orientations']
columns += [c for c in pair_effects if c.startswith('fdr_BH_')]
display(pair_effects[columns])


## Figuras de la comparación adicional
Cada figura muestra un selector y ámbito externo, usando pares únicos. No se añaden barras de incertidumbre inventadas. La concordancia del signo de una correlación diferencial no valida la orientación reguladora. Los nulos dirigidos originales no se reutilizan como nulos por pares únicos.


In [ ]:
import matplotlib.pyplot as plt
comparison = tables.get('intergate_vs_q_external')
if comparison is not None:
    figure_dir = OUTPUT_DIR / 'figures'
    figure_dir.mkdir(exist_ok=True)
    for (selector, scope), block in comparison.groupby(['selector', 'scope']):
        plot = block[block.unit.eq('unique_unordered_pairs')].set_index('method').loc[['InterGATE', 'q']]
        fig, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
        ax.scatter(plot.spearman_development_external, [1, 0], s=90)
        ax.set_yticks([1, 0], [f"InterGATE (n={int(plot.loc['InterGATE', 'n_effects'])})", f"q only (n={int(plot.loc['q', 'n_effects'])})"])
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-.6, 1.6)
        ax.set_xlabel('Spearman: development vs external effects')
        ax.set_title(f'{scope} | {selector} selector')
        ax.spines[['top', 'right']].set_visible(False)
        prepare_figure(fig, STYLE)
        for ext in ('pdf', 'svg', 'png'):
            fig.savefig(figure_dir / f'q_comparison_{selector}_{scope}.{ext}', bbox_inches='tight', dpi=STYLE['dpi'])
        plt.show()
else:
    print('Faltan paneles factoriales guardados; consulte q_comparator_input_status.csv.')


## Contraste pareado por pacientes (opcional; tampoco reentrena)
Remuestrea los mismos pacientes para ambos métodos, dentro de cohorte y grupo Basal-like/resto. Conserva preprocesado, efectos de desarrollo, paneles y pesos de cohortes. Antes de empezar exige reproducir recuentos y efectos originales de 08b. El p-valor es una aproximación bootstrap centrada, no una prueba exacta. No estima incertidumbre del entrenamiento ni de la selección. La opción queda desactivada por defecto.


In [ ]:
RUN_PATIENT_BOOTSTRAP = False
N_PATIENT_BOOTSTRAPS = 1000
BOOTSTRAP_SEED = 20260907
if RUN_PATIENT_BOOTSTRAP:
    from intergate.config import CFG
    from intergate.runtime_setup import prepare_runtime
    from intergate.review_external_bootstrap import run_patient_bootstrap
    runtime = prepare_runtime(CFG, include_external=True)
    paired = run_patient_bootstrap(runtime, RESULTS_ROOT, external_cohorts=tuple(CFG.TEST_COHORTS), n_bootstrap=N_PATIENT_BOOTSTRAPS, random_seed=BOOTSTRAP_SEED)
    display(paired)
else:
    print('Bootstrap por pacientes desactivado.')


## Figuras del manuscrito
En esta rama de GitHub, aplique primero `python tools/apply_cmpb_review.py --apply` desde la raíz; el ZIP de códigos revisados ya lleva el parche aplicado. Ejecute 10 para las figuras principales y 13 para S2/S3. Los tamaños están en `configs/publication_style.json`; las salidas nuevas terminan en `_large_fonts`.

**No vuelva a ejecutar 02, 06, 07, 09, las permutaciones de 12 ni 12c por este cambio.** No regenere el bloqueo original para ocultar modificaciones posteriores a la evaluación externa.
